# 🪨 AI-Based Rock Fragment Size Analysis System
### L&T EduTech — Open Pit Mining Operations
---
**Pipeline Overview:**
```
STEP 1 → Environment Setup & Imports
STEP 2 → Dataset Loading & Exploration
STEP 3 → Red Dot Detection & Scale Extraction
STEP 4 → Histogram & Texture Feature Extraction
STEP 5 → Build Training Dataset
STEP 6 → Train Histogram Calibration Model
STEP 7 → SAM Auto-Annotation
STEP 8 → Hybrid Inference (SAM + Roboflow)
STEP 9 → Fragment Size Measurement
STEP 10 → Export Results for Power BI
```

---
## STEP 1 — Environment Setup
Run this cell first. It installs all required libraries.

In [ ]:
# ── Install all required packages ─────────────────────────────────────
import subprocess, sys

packages = [
    'opencv-python',
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn',
    'scikit-image',
    'scipy',
    'Pillow',
    'joblib',
    'tqdm',
    'seaborn',
    'torch',
    'torchvision',
    'segment-anything'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed successfully')

In [ ]:
# ── Core Imports ───────────────────────────────────────────────────────
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from PIL                          import Image
from tqdm                         import tqdm
from scipy                        import stats
from skimage.feature              import graycomatrix, graycoprops
from sklearn.ensemble             import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection      import train_test_split, cross_val_score
from sklearn.preprocessing        import StandardScaler
from sklearn.metrics              import mean_absolute_error, r2_score
from sklearn.inspection           import permutation_importance

# Set plot style
plt.style.use('dark_background')
sns.set_palette('husl')

print('✅ All imports successful')
print(f'   OpenCV version  : {cv2.__version__}')
print(f'   NumPy  version  : {np.__version__}')
print(f'   Pandas version  : {pd.__version__}')

---
## STEP 2 — Dataset Loading & Exploration
⚠️ **Change `DATASET_PATH` to your actual folder path**

In [ ]:
# ── CONFIGURATION — Edit these paths ──────────────────────────────────
DATASET_PATH    = r'C:\Users\YourName\Desktop\rock_dataset'  # ← CHANGE THIS
OUTPUT_PATH     = r'C:\Users\YourName\Desktop\rock_output'   # ← CHANGE THIS
MODEL_SAVE_PATH = r'C:\Users\YourName\Desktop\rock_models'   # ← CHANGE THIS

# Supported image extensions
IMG_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']

# Red dot real diameter — set to None if unknown
# If you can measure even ONE image, set it here
ANCHOR_DOT_DIAMETER_MM = None  # e.g. 100 for 10cm ball

# Fragment size categories (RFSI based)
SIZE_CATEGORIES = {
    'Fine'    : (0.0,  0.5),
    'Small'   : (0.5,  2.0),
    'Medium'  : (2.0,  6.0),
    'Large'   : (6.0,  15.0),
    'Oversize': (15.0, float('inf'))
}

# Create output directories
os.makedirs(OUTPUT_PATH,     exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, 'annotated'),    exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, 'annotations'),  exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, 'csv_results'),  exist_ok=True)

print(f'✅ Configuration set')
print(f'   Dataset path : {DATASET_PATH}')
print(f'   Output path  : {OUTPUT_PATH}')

In [ ]:
# ── Load and Explore Dataset ───────────────────────────────────────────
def load_dataset(dataset_path):
    image_files = []
    for f in os.listdir(dataset_path):
        if any(f.lower().endswith(ext) for ext in IMG_EXTENSIONS):
            image_files.append(os.path.join(dataset_path, f))
    return sorted(image_files)

image_files = load_dataset(DATASET_PATH)
print(f'✅ Found {len(image_files)} images in dataset')

# Show dataset statistics
sizes, dims = [], []
for f in image_files:
    size = os.path.getsize(f) / (1024*1024)  # MB
    img  = cv2.imread(f)
    if img is not None:
        h, w = img.shape[:2]
        sizes.append(size)
        dims.append((w, h))

print(f'\n📊 Dataset Statistics:')
print(f'   Total images      : {len(image_files)}')
print(f'   Avg file size     : {np.mean(sizes):.2f} MB')
print(f'   Resolution range  : {min(dims)} to {max(dims)}')
print(f'   Total dataset size: {sum(sizes):.2f} MB')

In [ ]:
# ── Visualise Sample Images ────────────────────────────────────────────
n_samples = min(6, len(image_files))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < n_samples:
        img = cv2.imread(image_files[i])
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(f'Sample {i+1}: {os.path.basename(image_files[i])}',
                    fontsize=9, color='white')
        ax.axis('off')
    else:
        ax.axis('off')

plt.suptitle('Dataset Sample Images', fontsize=14, color='white', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'dataset_samples.png'),
            bbox_inches='tight', dpi=150)
plt.show()
print('✅ Sample visualisation saved')

---
## STEP 3 — Red Dot Detection & Scale Extraction
Detects red circular markers and extracts their pixel diameter as scale reference.

In [ ]:
# ── Red Dot Detector Function ──────────────────────────────────────────
def detect_red_dots(image, min_area=300, circularity_thresh=0.6):
    """
    Detects red circular markers in image.
    Returns list of detected dots with their properties.
    """
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Red wraps around 0/180 in HSV — need two ranges
    lower_red_1 = np.array([0,   120,  70])
    upper_red_1 = np.array([10,  255, 255])
    lower_red_2 = np.array([160, 120,  70])
    upper_red_2 = np.array([180, 255, 255])

    mask1    = cv2.inRange(hsv, lower_red_1, upper_red_1)
    mask2    = cv2.inRange(hsv, lower_red_2, upper_red_2)
    red_mask = cv2.bitwise_or(mask1, mask2)

    # Morphological cleanup
    kernel   = np.ones((7, 7), np.uint8)
    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN,  kernel)
    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    dots = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        # Circularity check: 4π×area / perimeter²
        perimeter    = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        circularity  = (4 * np.pi * area) / (perimeter ** 2)

        if circularity < circularity_thresh:
            continue  # not circular enough → not a ball

        (cx, cy), radius = cv2.minEnclosingCircle(cnt)
        dots.append({
            'center'          : (int(cx), int(cy)),
            'radius_px'       : float(radius),
            'diameter_px'     : float(radius * 2),
            'area_px'         : float(area),
            'circularity'     : float(circularity)
        })

    # Sort by area descending
    dots = sorted(dots, key=lambda x: x['area_px'], reverse=True)
    return dots, red_mask

print('✅ Red dot detector function defined')

In [ ]:
# ── Test Red Dot Detection on First Image ─────────────────────────────
test_img_path = image_files[0]
test_img      = cv2.imread(test_img_path)
test_img_rgb  = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

dots, red_mask = detect_red_dots(test_img)

# Draw detections
vis_img = test_img_rgb.copy()
for i, dot in enumerate(dots):
    cx, cy = dot['center']
    r      = int(dot['radius_px'])
    cv2.circle(vis_img, (cx, cy), r,       (0, 255, 0), 3)
    cv2.circle(vis_img, (cx, cy), 5,       (255, 255, 0), -1)
    cv2.putText(vis_img,
        f"D={dot['diameter_px']:.1f}px",
        (cx + r + 5, cy),
        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2
    )

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(test_img_rgb)
axes[0].set_title('Original Image', color='white')
axes[0].axis('off')

axes[1].imshow(red_mask, cmap='hot')
axes[1].set_title('Red Mask (HSV Detection)', color='white')
axes[1].axis('off')

axes[2].imshow(vis_img)
axes[2].set_title(f'Detected {len(dots)} Red Dot(s)', color='white')
axes[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'red_dot_detection_test.png'),
            bbox_inches='tight', dpi=150)
plt.show()

print(f'\n📍 Detected {len(dots)} red dot(s):')
for i, dot in enumerate(dots):
    print(f'   Dot {i+1}: center={dot["center"]}, '
          f'diameter={dot["diameter_px"]:.1f}px, '
          f'circularity={dot["circularity"]:.3f}')

In [ ]:
# ── Batch Red Dot Detection — All Images ──────────────────────────────
dot_records = []
failed      = []

for img_path in tqdm(image_files, desc='Detecting red dots'):
    img  = cv2.imread(img_path)
    if img is None:
        failed.append(img_path)
        continue

    dots, _ = detect_red_dots(img)

    if not dots:
        failed.append(img_path)
        dot_records.append({
            'image_path'        : img_path,
            'image_name'        : os.path.basename(img_path),
            'dots_found'        : 0,
            'avg_diameter_px'   : None,
            'dot_diameters'     : [],
            'reference_area_px' : None
        })
        continue

    diameters = [d['diameter_px'] for d in dots]
    areas     = [d['area_px']     for d in dots]

    dot_records.append({
        'image_path'        : img_path,
        'image_name'        : os.path.basename(img_path),
        'dots_found'        : len(dots),
        'avg_diameter_px'   : np.mean(diameters),
        'dot_diameters'     : diameters,
        'reference_area_px' : np.mean(areas)
    })

dot_df = pd.DataFrame(dot_records)
dot_df.to_csv(os.path.join(OUTPUT_PATH, 'csv_results', 'dot_detections.csv'),
              index=False)

detected = dot_df[dot_df['dots_found'] > 0]
print(f'\n📊 Red Dot Detection Summary:')
print(f'   Images processed    : {len(dot_df)}')
print(f'   Dots found          : {len(detected)} images')
print(f'   No dots detected    : {len(failed)} images')
print(f'   Avg dot diameter    : {detected["avg_diameter_px"].mean():.1f} px')
print(f'   Diameter range      : {detected["avg_diameter_px"].min():.1f} — '
      f'{detected["avg_diameter_px"].max():.1f} px')
print(f'\n✅ Results saved to dot_detections.csv')

---
## STEP 4 — Histogram & Texture Feature Extraction
This is the core of your novel contribution — learning scale from visual statistics.

In [ ]:
# ── Feature Extraction Function ────────────────────────────────────────
def extract_features(image):
    """
    Extracts histogram, texture, and frequency features.
    These form the INPUT to the calibration model.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    features = {}

    # ── 1. Basic Intensity Statistics ─────────────────────
    features['mean_intensity']  = float(np.mean(gray))
    features['std_intensity']   = float(np.std(gray))
    features['skewness']        = float(stats.skew(gray.flatten()))
    features['kurtosis']        = float(stats.kurtosis(gray.flatten()))
    features['min_intensity']   = float(np.min(gray))
    features['max_intensity']   = float(np.max(gray))
    features['dynamic_range']   = float(np.max(gray) - np.min(gray))

    # ── 2. Histogram Features ─────────────────────────────
    hist     = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    hist_n   = hist / (hist.sum() + 1e-10)  # normalised
    features['hist_entropy']    = float(stats.entropy(hist_n + 1e-10))
    features['hist_uniformity'] = float(np.sum(hist_n ** 2))

    # Histogram percentiles
    for p in [10, 25, 50, 75, 90]:
        features[f'percentile_{p}'] = float(np.percentile(gray, p))

    # ── 3. GLCM Texture Features ──────────────────────────
    # Captures spatial pixel relationships → rock size proxy
    gray_8 = (gray).astype(np.uint8)
    glcm   = graycomatrix(
        gray_8, [1, 2, 4, 8],
        [0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=64, symmetric=True, normed=True
    )
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
        features[f'glcm_{prop}'] = float(graycoprops(glcm, prop).mean())

    # ── 4. FFT Frequency Features ─────────────────────────
    # Large rocks → low frequency, Fine gravel → high frequency
    fft       = np.fft.fft2(gray)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.log(np.abs(fft_shift) + 1)
    h, w      = magnitude.shape
    cy, cx    = h // 2, w // 2

    for r in [20, 50, 100]:
        y1, y2 = max(0,cy-r), min(h,cy+r)
        x1, x2 = max(0,cx-r), min(w,cx+r)
        features[f'fft_energy_r{r}'] = float(magnitude[y1:y2, x1:x2].mean())

    features['fft_total_energy'] = float(magnitude.mean())
    features['fft_freq_ratio']   = float(
        features['fft_energy_r20'] / (features['fft_total_energy'] + 1e-10)
    )

    # ── 5. Multi-Scale Histogram (Image Pyramid) ──────────
    # Different scales capture fragment size at different resolutions
    for scale, name in [(0.5, 'half'), (0.25, 'quarter')]:
        h_s   = max(1, int(gray.shape[0] * scale))
        w_s   = max(1, int(gray.shape[1] * scale))
        small = cv2.resize(gray, (w_s, h_s))
        features[f'mean_{name}_scale']  = float(np.mean(small))
        features[f'std_{name}_scale']   = float(np.std(small))
        hist_s = cv2.calcHist([small],[0],None,[64],[0,256]).flatten()
        hist_s = hist_s / (hist_s.sum() + 1e-10)
        features[f'entropy_{name}_scale'] = float(stats.entropy(hist_s + 1e-10))

    # ── 6. Edge Density (Laplacian) ───────────────────────
    # High edge density → many small fragments
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    features['laplacian_var']    = float(laplacian.var())
    features['laplacian_mean']   = float(np.abs(laplacian).mean())

    # ── 7. HSV Color Features ─────────────────────────────
    for i, ch in enumerate(['hue', 'sat', 'val']):
        features[f'{ch}_mean'] = float(np.mean(hsv[:,:,i]))
        features[f'{ch}_std']  = float(np.std(hsv[:,:,i]))

    return features

print('✅ Feature extractor defined')
print(f'   Features per image: ~{len(extract_features(cv2.imread(image_files[0])))} features')

In [ ]:
# ── Visualise Feature Distributions ───────────────────────────────────
sample_features = []
for img_path in image_files[:20]:  # sample 20 images
    img  = cv2.imread(img_path)
    feat = extract_features(img)
    feat['image'] = os.path.basename(img_path)
    sample_features.append(feat)

feat_df = pd.DataFrame(sample_features)

# Plot key features
key_features = ['mean_intensity', 'std_intensity', 'hist_entropy',
                'glcm_contrast', 'laplacian_var', 'fft_freq_ratio']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    axes[i].hist(feat_df[feat], bins=15,
                color='#00d4ff', edgecolor='white', alpha=0.8)
    axes[i].set_title(feat, color='white', fontsize=10)
    axes[i].set_xlabel('Value', color='grey')
    axes[i].set_ylabel('Count', color='grey')

plt.suptitle('Feature Distributions Across Sample Images',
             color='white', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'feature_distributions.png'),
            bbox_inches='tight', dpi=150)
plt.show()
print('✅ Feature distribution plot saved')

---
## STEP 5 — Build Training Dataset
Combine red dot scale (ground truth) + histogram features (input) → training pairs.

In [ ]:
# ── Build Training Dataset ─────────────────────────────────────────────
training_records = []
skipped          = []

for img_path in tqdm(image_files, desc='Building training data'):
    img = cv2.imread(img_path)
    if img is None:
        skipped.append(img_path)
        continue

    # ── Ground Truth: scale from red dots ─────────────────
    dots, _ = detect_red_dots(img)
    if not dots:
        skipped.append(img_path)
        continue

    avg_diameter_px  = np.mean([d['diameter_px'] for d in dots])
    avg_area_px      = np.mean([d['area_px']     for d in dots])

    # ── Input: histogram features ──────────────────────────
    features = extract_features(img)

    # ── Combine ────────────────────────────────────────────
    record = features.copy()
    record['image_path']         = img_path
    record['image_name']         = os.path.basename(img_path)
    record['target_diameter_px'] = avg_diameter_px
    record['target_area_px']     = avg_area_px
    record['n_dots']             = len(dots)

    # Absolute scale if anchor known
    if ANCHOR_DOT_DIAMETER_MM:
        record['mm_per_pixel'] = ANCHOR_DOT_DIAMETER_MM / avg_diameter_px
    else:
        record['mm_per_pixel'] = None

    training_records.append(record)

train_df = pd.DataFrame(training_records)
train_df.to_csv(
    os.path.join(OUTPUT_PATH, 'csv_results', 'training_dataset.csv'),
    index=False
)

print(f'\n📊 Training Dataset Summary:')
print(f'   Total training samples : {len(train_df)}')
print(f'   Skipped (no dots)      : {len(skipped)}')
print(f'   Feature columns        : {len([c for c in train_df.columns if c not in ["image_path","image_name","target_diameter_px","target_area_px","n_dots","mm_per_pixel"]])}')
print(f'\n✅ Training dataset saved to training_dataset.csv')

---
## STEP 6 — Train Histogram Calibration Model
This is your novel contribution — predicting scale from image statistics alone.

In [ ]:
# ── Train Calibration Model ────────────────────────────────────────────
EXCLUDE_COLS = ['image_path', 'image_name', 'target_diameter_px',
                'target_area_px', 'n_dots', 'mm_per_pixel']
feature_cols = [c for c in train_df.columns if c not in EXCLUDE_COLS]

X = train_df[feature_cols].values.astype(np.float32)
y = train_df['target_diameter_px'].values.astype(np.float32)

# Handle NaN values
X = np.nan_to_num(X, nan=0.0)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ── Model 1: Random Forest ─────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200, max_depth=12,
    min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred  = rf.predict(X_test)
rf_mae   = mean_absolute_error(y_test, rf_pred)
rf_r2    = r2_score(y_test, rf_pred)

# ── Model 2: Gradient Boosting ─────────────────────────
gb = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05,
    max_depth=5,      random_state=42
)
gb.fit(X_train, y_train)
gb_pred  = gb.predict(X_test)
gb_mae   = mean_absolute_error(y_test, gb_pred)
gb_r2    = r2_score(y_test, gb_pred)

print('📊 Model Performance:')
print(f'   Random Forest   → MAE: {rf_mae:.2f}px, R²: {rf_r2:.4f}')
print(f'   Gradient Boost  → MAE: {gb_mae:.2f}px, R²: {gb_r2:.4f}')

# Pick best model
if rf_r2 >= gb_r2:
    best_model, best_name = rf, 'RandomForest'
else:
    best_model, best_name = gb, 'GradientBoosting'

print(f'\n🏆 Best model: {best_name} (R²={max(rf_r2, gb_r2):.4f})')

# Save everything
joblib.dump(best_model,  os.path.join(MODEL_SAVE_PATH, 'calibration_model.pkl'))
joblib.dump(scaler,      os.path.join(MODEL_SAVE_PATH, 'feature_scaler.pkl'))
joblib.dump(feature_cols,os.path.join(MODEL_SAVE_PATH, 'feature_columns.pkl'))

print('\n✅ Models saved to disk')

In [ ]:
# ── Model Evaluation Plots ─────────────────────────────────────────────
best_pred = best_model.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Predicted vs Actual
axes[0].scatter(y_test, best_pred, alpha=0.7,
                color='#00d4ff', edgecolors='white', s=60)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Diameter (px)', color='grey')
axes[0].set_ylabel('Predicted Diameter (px)', color='grey')
axes[0].set_title(f'{best_name}\nPredicted vs Actual', color='white')
axes[0].legend()

# 2. Residuals
residuals = best_pred - y_test
axes[1].hist(residuals, bins=20,
             color='#ff6b6b', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='white', linestyle='--')
axes[1].set_xlabel('Residual (px)', color='grey')
axes[1].set_ylabel('Count', color='grey')
axes[1].set_title('Prediction Residuals', color='white')

# 3. Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    top_n       = 10
    top_idx     = np.argsort(importances)[-top_n:]
    top_names   = [feature_cols[i] for i in top_idx]
    top_vals    = importances[top_idx]

    axes[2].barh(top_names, top_vals,
                color='#7bed9f', edgecolor='white')
    axes[2].set_xlabel('Importance', color='grey')
    axes[2].set_title('Top 10 Feature Importances', color='white')

plt.suptitle('Histogram Calibration Model Evaluation',
             color='white', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'model_evaluation.png'),
            bbox_inches='tight', dpi=150)
plt.show()

print(f'\n📊 Final Model Metrics:')
print(f'   MAE  : {mean_absolute_error(y_test, best_pred):.2f} px')
print(f'   R²   : {r2_score(y_test, best_pred):.4f}')
print(f'   RMSE : {np.sqrt(np.mean((y_test - best_pred)**2)):.2f} px')

---
## STEP 7 — RFSI Based Fragment Analysis
Compute Relative Fragment Size Index for all images using detected red dots.

In [ ]:
# ── RFSI Fragment Analyser ─────────────────────────────────────────────
def classify_rfsi(rfsi):
    for category, (low, high) in SIZE_CATEGORIES.items():
        if low <= rfsi < high:
            return category
    return 'Unknown'

def analyse_image_fragments(image_path,
                            calibration_model=None,
                            scaler=None,
                            feature_cols=None):
    """
    Full fragment analysis on one image.
    Uses red dots if present, histogram model if not.
    """
    img      = cv2.imread(image_path)
    img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w     = img.shape[:2]
    result   = {'image_path': image_path,
                'image_name': os.path.basename(image_path)}

    # ── Scale Reference ────────────────────────────────────
    dots, _  = detect_red_dots(img)

    if dots:
        ref_area_px = np.mean([d['area_px'] for d in dots])
        ref_diam_px = np.mean([d['diameter_px'] for d in dots])
        scale_method = 'red_dot'
    elif calibration_model is not None:
        # Fallback: histogram model predicts scale
        features = extract_features(img)
        X_new    = np.array([[features[c] for c in feature_cols]])
        X_new    = np.nan_to_num(X_new)
        X_new    = scaler.transform(X_new)
        ref_diam_px  = calibration_model.predict(X_new)[0]
        ref_area_px  = np.pi * (ref_diam_px / 2) ** 2
        scale_method = 'histogram_model'
    else:
        result['error'] = 'No scale reference available'
        return result

    result['ref_diameter_px'] = ref_diam_px
    result['scale_method']    = scale_method

    # ── Sharpness Check ────────────────────────────────────
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    result['sharpness_score'] = float(sharpness)
    result['image_quality']   = 'good' if sharpness > 100 else 'blurry'

    # ── Basic Fragment Detection via Edge + Contours ───────
    # (Replace contours with SAM/Roboflow masks in STEP 8)
    blurred  = cv2.GaussianBlur(gray, (5,5), 0)
    edges    = cv2.Canny(blurred, 50, 150)
    kernel   = np.ones((3,3), np.uint8)
    dilated  = cv2.dilate(edges, kernel, iterations=2)

    contours, _ = cv2.findContours(
        dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Filter and classify fragments
    fragments = []
    vis_img   = img_rgb.copy()

    CATEGORY_COLORS = {
        'Fine'    : (100, 200, 255),
        'Small'   : (100, 255, 150),
        'Medium'  : (255, 220, 100),
        'Large'   : (255, 140, 50),
        'Oversize': (255, 60,  60)
    }

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 500:  # ignore tiny noise
            continue

        rfsi     = area / (ref_area_px + 1e-10)
        category = classify_rfsi(rfsi)
        color    = CATEGORY_COLORS.get(category, (200,200,200))

        # Bounding box
        x, y, bw, bh = cv2.boundingRect(cnt)

        # Draw on visualisation
        cv2.drawContours(vis_img, [cnt], -1, color, 2)

        fragments.append({
            'area_px'  : float(area),
            'rfsi'     : float(rfsi),
            'category' : category,
            'bbox'     : [int(x), int(y), int(bw), int(bh)]
        })

    result['fragments']       = fragments
    result['total_fragments'] = len(fragments)
    result['vis_image']       = vis_img

    # Category distribution
    for cat in SIZE_CATEGORIES:
        count = sum(1 for f in fragments if f['category'] == cat)
        result[f'count_{cat}'] = count
        result[f'pct_{cat}']   = round(
            100 * count / max(len(fragments), 1), 1
        )

    return result

print('✅ Fragment analyser defined')

In [ ]:
# ── Run Full Analysis on All Images ───────────────────────────────────
# Load saved model
cal_model    = joblib.load(os.path.join(MODEL_SAVE_PATH, 'calibration_model.pkl'))
cal_scaler   = joblib.load(os.path.join(MODEL_SAVE_PATH, 'feature_scaler.pkl'))
cal_features = joblib.load(os.path.join(MODEL_SAVE_PATH, 'feature_columns.pkl'))

all_results  = []

for img_path in tqdm(image_files, desc='Analysing fragments'):
    res = analyse_image_fragments(
        img_path, cal_model, cal_scaler, cal_features
    )

    # Save annotated image
    if 'vis_image' in res:
        out_name = 'annotated_' + os.path.basename(img_path)
        out_path = os.path.join(OUTPUT_PATH, 'annotated', out_name)
        cv2.imwrite(out_path,
                    cv2.cvtColor(res['vis_image'], cv2.COLOR_RGB2BGR))
        del res['vis_image']   # don't store image array in dataframe
        del res['fragments']   # stored separately if needed

    all_results.append(res)

results_df = pd.DataFrame(all_results)
results_df.to_csv(
    os.path.join(OUTPUT_PATH, 'csv_results', 'fragment_analysis_results.csv'),
    index=False
)

print(f'\n📊 Analysis Complete:')
print(f'   Images analysed    : {len(results_df)}')
print(f'   Avg fragments/img  : {results_df["total_fragments"].mean():.1f}')
print(f'   Scale via red dot  : {(results_df["scale_method"]=="red_dot").sum()}')
print(f'   Scale via model    : {(results_df["scale_method"]=="histogram_model").sum()}')
print(f'\n✅ Results saved to fragment_analysis_results.csv')

---
## STEP 8 — Results Visualisation & Power BI Export
Generate charts and export clean CSV for Power BI dashboard.

In [ ]:
# ── Overall Fragment Size Distribution ────────────────────────────────
category_totals = {}
for cat in SIZE_CATEGORIES:
    col = f'count_{cat}'
    if col in results_df.columns:
        category_totals[cat] = results_df[col].sum()

COLORS = {
    'Fine'    : '#64c8ff',
    'Small'   : '#64ff96',
    'Medium'  : '#ffdc64',
    'Large'   : '#ff8c32',
    'Oversize': '#ff3c3c'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Pie chart
labels = list(category_totals.keys())
values = list(category_totals.values())
colors = [COLORS[l] for l in labels]

axes[0].pie(values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            textprops={'color': 'white'})
axes[0].set_title('Fragment Size Distribution\n(All Images)', color='white')

# 2. Bar chart per image
x     = np.arange(len(results_df))
width = 0.15
for i, cat in enumerate(SIZE_CATEGORIES):
    col = f'pct_{cat}'
    if col in results_df.columns:
        axes[1].bar(x + i*width,
                   results_df[col],
                   width, label=cat,
                   color=COLORS[cat], alpha=0.85)

axes[1].set_xlabel('Image Index', color='grey')
axes[1].set_ylabel('Percentage (%)', color='grey')
axes[1].set_title('Size Distribution Per Image', color='white')
axes[1].legend(loc='upper right', fontsize=8)

# 3. Oversize alert bar
if 'pct_Oversize' in results_df.columns:
    colors_alert = ['#ff3c3c' if v > 20 else '#64ff96'
                    for v in results_df['pct_Oversize']]
    axes[2].bar(x, results_df['pct_Oversize'],
               color=colors_alert, edgecolor='white', alpha=0.9)
    axes[2].axhline(20, color='yellow', linestyle='--',
                    label='Alert Threshold (20%)')
    axes[2].set_xlabel('Image Index', color='grey')
    axes[2].set_ylabel('Oversize %', color='grey')
    axes[2].set_title('Oversize Fragment Alert\n(Red = Above Threshold)',
                      color='white')
    axes[2].legend()

plt.suptitle('Rock Fragment Size Analysis Results',
             color='white', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'fragment_analysis_charts.png'),
            bbox_inches='tight', dpi=150)
plt.show()
print('✅ Result charts saved')

In [ ]:
# ── Power BI Export ────────────────────────────────────────────────────
import datetime

powerbi_df = results_df[[
    'image_name',
    'total_fragments',
    'scale_method',
    'sharpness_score',
    'image_quality',
    'ref_diameter_px'
] + [f'count_{c}' for c in SIZE_CATEGORIES if f'count_{c}' in results_df.columns]
  + [f'pct_{c}'   for c in SIZE_CATEGORIES if f'pct_{c}'   in results_df.columns]
].copy()

# Add metadata for Power BI
powerbi_df['blast_zone_id']  = powerbi_df['image_name'].str.extract(r'(\d+)')
powerbi_df['timestamp']      = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
powerbi_df['oversize_alert'] = powerbi_df.get(
    'pct_Oversize', pd.Series([0]*len(powerbi_df))
).apply(lambda x: 'ALERT' if x > 20 else 'OK')

# Save Power BI ready CSV
pb_path = os.path.join(OUTPUT_PATH, 'csv_results', 'powerbi_dashboard_data.csv')
powerbi_df.to_csv(pb_path, index=False)

print('📊 Power BI Export Summary:')
print(powerbi_df[['image_name', 'total_fragments',
                   'pct_Oversize', 'oversize_alert',
                   'scale_method']].head(10).to_string())
print(f'\n✅ Power BI CSV saved to: {pb_path}')
print('   → Open Power BI Desktop')
print('   → Get Data → Text/CSV → Select powerbi_dashboard_data.csv')
print('   → Build histogram, oversize alerts, and zone comparison charts')

In [ ]:
# ── DEPLOYMENT: Predict Scale Without Any Red Dot ─────────────────────
def deploy_no_marker(image_path):
    """
    Real-world deployment function.
    No red dot needed — histogram model predicts scale.
    """
    print(f'\n🔍 Analysing: {os.path.basename(image_path)}')

    img = cv2.imread(image_path)
    if img is None:
        print('   ❌ Could not load image')
        return None

    # Sharpness check
    gray      = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    print(f'   Sharpness score : {sharpness:.1f} '
          f'({"✅ Good" if sharpness > 100 else "⚠️ Blurry"})')

    # Try red dot first
    dots, _ = detect_red_dots(img)
    if dots:
        ref_diam = np.mean([d['diameter_px'] for d in dots])
        ref_area = np.mean([d['area_px']     for d in dots])
        method   = '🔴 Red Dot'
    else:
        # Histogram model fallback
        features = extract_features(img)
        X_new    = np.array([[features[c] for c in cal_features]])
        X_new    = np.nan_to_num(X_new)
        X_new    = cal_scaler.transform(X_new)
        ref_diam = cal_model.predict(X_new)[0]
        ref_area = np.pi * (ref_diam / 2) ** 2
        method   = '📊 Histogram Model'

    print(f'   Scale method    : {method}')
    print(f'   Reference diam  : {ref_diam:.1f} px')

    # Run analysis
    result = analyse_image_fragments(
        image_path, cal_model, cal_scaler, cal_features
    )

    print(f'   Fragments found : {result["total_fragments"]}')
    for cat in SIZE_CATEGORIES:
        pct = result.get(f'pct_{cat}', 0)
        bar = '█' * int(pct / 5)
        print(f'   {cat:<10}: {pct:5.1f}% {bar}')

    if result.get('pct_Oversize', 0) > 20:
        print('\n   🚨 OVERSIZE ALERT — Crusher blockage risk!')

    return result

# Test on first image
test_result = deploy_no_marker(image_files[0])
print('\n✅ Deployment function working correctly')

---
## ✅ Pipeline Complete

### What Was Built:
| Step | Component | Output |
|------|-----------|--------|
| 1 | Environment Setup | All libraries installed |
| 2 | Dataset Loading | Dataset statistics & samples |
| 3 | Red Dot Detection | Scale per image (HSV detection) |
| 4 | Feature Extraction | Histogram + GLCM + FFT features |
| 5 | Training Dataset | feature-scale pairs CSV |
| 6 | Calibration Model | RandomForest / GradientBoosting |
| 7 | Fragment Analysis | RFSI per rock + size categories |
| 8 | Power BI Export | Dashboard-ready CSV |

### Output Files:
```
rock_output/
├── annotated/                  ← visualised images
├── csv_results/
│   ├── dot_detections.csv
│   ├── training_dataset.csv
│   ├── fragment_analysis_results.csv
│   └── powerbi_dashboard_data.csv  ← load this in Power BI
├── dataset_samples.png
├── feature_distributions.png
├── model_evaluation.png
└── fragment_analysis_charts.png

rock_models/
├── calibration_model.pkl
├── feature_scaler.pkl
└── feature_columns.pkl
```

### Next Steps:
1. **Integrate SAM** for pixel-accurate segmentation (replace Canny contours in Step 7)
2. **Add Roboflow model** for rock classification
3. **Connect Power BI** to the CSV output
4. **Collect more images** to improve calibration model accuracy